![ATARRI logos](img/logos.png)

## 4.2 Interpolating modelled AOD with AERONET observations

The first step towards evaluating observations and model data is to interpolate them, so that they are matched spatially and temporally for further analysis. For this, we will use Providentia's Interpolation mode.

We will also use AERONET AOD data from two algorithms: Direct Sun v3 Lev 1.5 and O'Neill v3 Lev 1.5.

## Observational data pre-processing

Before we interpolate, we need to check whether our observation files have the same temporal frequency as our model files, else Providentia will not work.

Our downloaded observation data from AERONET are currently .html files, found in the `/shared/data/obs/nonghost/nasa-aeronet/{algorithm}/original_files/` folder, and by inspecting them we can see they have data every few minutes. Therefore, we will have to resample the files to a 3-hourly frequency.

For the Direct Sun evaluation, we want two datasets: total AOD at 550nm and the Angstrom exponent at 440-870 nm. However, the files do not contain total AOD at this wavelength, so we need to spectrally calculate it based off the currently available wavelenghts: 440nm, 675nm and 870nm.

For the O'Neill evaluation, we want coarse mode AOD at 500nm. Despite not being at the same wavelength as our dust models, we will take it as comparable for our analysis.

Therefore, to perform these operations and convert the data files to a .nc output, we will run the `../scripts/VT4_Aeronetv3.py` script. This will output files into the `directsun_v3-lev15/` and `oneill_v3-lev15/` folders, which are found in the `/shared/trainees/user/obs/nonghost/nasa-aeronet`. 

* Tip: To obtain your own version of the post-processed datasets, you need to modify lines 505, 507, 665 and 667 of the script with your own user name (atarriXX).

Finally, to run the script you need to input the start date, end date, resample frequency (`3H`), AERONET version (`on` or `ds`) and data level (`15`)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
%run ../scripts/VT4_Aeronetv3.py -s "20250301" -e "20250309" -f "3H" -v "on" -l "15"

['../scripts/VT4_Aeronetv3.py', '-s', '20250301', '-e', '20250309', '-f', '3H', '-v', 'on', '-l', '15']
Running formatting for follwing variables: Startdate - 20250301, Enddate - 20250309, Frequency - 3H, Aeronet Variable - on, level - 15
3H
20250301
Formatting for date 20250301 ...
File /shared/data/obs/nonghost/nasa-aeronet/oneill_v3-lev15/original_files/20250301.html
Station file: ../scripts/aeronet_locations_v3_daily.txt
AFTER 5000
Setting variables ...
NDATA
[[-100. -100. -100. ... -100. -100. -100.]
 [-100. -100. -100. ... -100. -100. -100.]
 [-100. -100. -100. ... -100. -100. -100.]
 ...
 [-100. -100. -100. ... -100. -100. -100.]
 [-100. -100. -100. ... -100. -100. -100.]
 [-100. -100. -100. ... -100. -100. -100.]]
(8, 5000)
date 20250301 MFILES ['0', '3', '5', '2', '1']
202503 od500aero /shared/trainees/tvintimi/obs/nonghost/nasa-aeronet/oneill_v3-lev15/3hourly/od500aero
MTS SLICES
slice(0, 8, None)
Updating variable od500aero (shape (248, 5000)) with shape (40000,) from tmp wi

In [3]:
%run ../scripts/VT4_Aeronetv3.py -s "20250301" -e "20250309" -f "3H" -v "ds" -l "15"

['../scripts/VT4_Aeronetv3.py', '-s', '20250301', '-e', '20250309', '-f', '3H', '-v', 'ds', '-l', '15']
Running formatting for follwing variables: Startdate - 20250301, Enddate - 20250309, Frequency - 3H, Aeronet Variable - ds, level - 15
3H
20250301
Formatting for date 20250301 ...
File /shared/data/obs/nonghost/nasa-aeronet/directsun_v3-lev15/original_files/20250301.html
Station file: ../scripts/aeronet_locations_v3_daily.txt
AFTER 5000
                Date(dd:mm:yyyy)  Time(hh:mm:ss)  AOD_870nm  AOD_675nm  \
AERONET_Site                                                             
AAU_Jackros_ET                74              74         74         74   
ARM_BNF                       33              33         33         33   
ARM_CRG                       56              56         56         56   
ARM_Graciosa                  25              25         25         25   
ARM_KCG                        3               3          3          3   
...                          ...       

We now have obtained the following variables from the Direct Sun data, each in their own .nc file and stored in the `3hourly/` subfolder:

- AOD at 550nm
- Total aerosol Angstrom parameter [470-870]

Likewise, for O'Neill data we have obtained the following variables, each in their own .nc file:

- AOD at 500nm (not used for our interpolation)
- Coarse AOD at 500nm
- Fine AOD at 500nm (not used for our interpolation)

## Interpolation mode requirements

Now that we have our base files, we will interpolate. We will use the `od550aero` and `od500aerocoarse` AERONET files together with the non-interpolated MONARCH forecasts for March 2025, which are found in `/shared/data/exp_to_interp/monarch/regional/3hourly/od550_dust`.

In our Providentia directory, we have our own configuration file: `atarri_interpolation.conf`, inside which of we will create two sections. We need to make sure the fields in each section have been populated as follows:

- `network`: Direct Sun for `od550aero` and O'Neill for `od500aerocoarse`.
- `species`: `od550aero` and `od500aerocoarse`.
- `start_date` and `end_date`: We will interpolate for all of March 2025. Note that in Providentia, the day selected in `end_date` is not included.
- `experiment`, `experiments` or `model`: We will interpolate `monarch`.
- `domain`: We need to set it as `regional`.

![Interpolation configuration](img/30_interp-conf.png)

For the interpolation to succeed, we need to make sure our paths in `..providentia/settings/data_paths.yaml` are correctly written. You will currently have something like this:

![Data paths](img/10_data-paths.png)

These roots need to be changed to the following (where atarriXX is your Jupyter username):

~~~
ghost_root: /shared/trainees/atarriXX/obs/ghost
mod_root: /shared/trainees/atarriXX/mod
mod_to_interp_root: /shared/data/exp_to_interp
nonghost_root: /shared/trainees/atarriXX/obs/nonghost
~~~

Remember, our non-interpolated model data is stored in the `/shared` partition, and our observations and output interpolated data will go inside our individual user paths in `/shared/trainees/`.

You may have also noticed that we have different species names for our observations (`od550aero` and `od500aerocoarse`) and model (`od550_dust`). This is because, for our evaluation, we are comparing dust as a component of total AOD. In order to read and interpolate them, we need to go to `../providentia/settings/internal/mapping_species.yaml`. This .yaml file functions as a sort of dictionary, where we can match the names of our observation species and model species. Under `od550aero` and `od500aerocoarse`, we need to set `od550_dust` (the variable name from our dust model files).

![Mapping species](img/11_mapping-species.png)

## Running Interpolation

To run an interpolation of our AERONET and MONARCH files, we will run the following lines inside our terminal, one for each interpolation:

`./bin/providentia --interp --config=atarri_interpolation.conf --section=DS`

`./bin/providentia --interp --config=atarri_interpolation.conf --section=ON`

Each time we run the commands we will see print statements with the input configurations along with details of the submission.

![Submission of conf file](img/07_conf-submission.png)

Providentia also produces logs every time we interpolate. If you want to check these out, you can go to the `logs/interpolation/management_logs` directory and look for the corresponding .out file. This may be useful in case you ever run into errors while trying to interpolate.

![Submission log](img/08_submission-log.png)

Since our interpolation was successful, we can double check our output in the `/shared/trainees/atarriXX/mod/1.5/monarch-regional-000/3hourly` directory, which has been newly with the structure of our input .conf. Here, we see our correctly interpolated models for both AOD at 550nm and coarse AOD at 500nm.


![Interpolated file](img/09_interpolated-file.png)

## References and further reading

- AERONET [website](https://aeronet.gsfc.nasa.gov/)
- AERONET [Aerosol Optical Depth](https://aeronet.gsfc.nasa.gov/new_web/aerosols.html)
- AERONET [AOD data download tool](https://aeronet.gsfc.nasa.gov/new_web/webtool_aod_v3.html)
- EUMETSAT Dust Aerosol Detection, Monitoring and Forecasting course: [AERONET section](https://dust.trainhub.eumetsat.int/docs/aeronet.html)

For more information on Interpolation mode, please check out the [Interpolation page](https://providentia.readthedocs.io/en/latest/Interpolation.html) of the Providentia docs.

Providentia is an internal tool developed at BSC-CNS. For any issues, please contact the active developers: [Dene Bowdalo](mailto:dene.bowdalo@bsc.es), [Alba Vilanova](mailto:alba.vilanova@bsc.es) and [Paula Serrano](mailto:paula.serrano@bsc.es).

<table style="width:100%;">
    <tr>
        <td style="text-align: center;"><a href="VT4-1-intro_Providentia.ipynb" style="font-size: 18px;">⬅ Previous</a></td>
        <td style="text-align: center;"><a href="VT4-3-dashboard_mode-od550du.ipynb" style="font-size: 18px;">Next ➡</a></td>
    </tr>
</table>